In [3]:
import os
import shutil

import numpy as np
import pandas as pd
import yaml
from tqdm.contrib.concurrent import thread_map
from ultralytics import YOLO

Example of project yaml:
```yaml
# Ultralytics YOLO 🚀, AGPL-3.0 license
# COCO128 dataset https://www.kaggle.com/ultralytics/coco128 (first 128 images from COCO train2017) by Ultralytics
# Example usage: yolo train data=coco128.yaml
# parent
# ├── ultralytics
# └── datasets
#     └── coco128  ← downloads here (7 MB)


# Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
path: ../datasets/position_db_pose  # dataset root dir
train: images/train  # train images (relative to 'path') 128 images
val: images/val  # val images (relative to 'path') 128 images
test: images/test  # test images (optional)

kpt_shape: [1, 2]

# Classes
names:
  0: person
```

Example of annotaion txt file
```txt
0 0.726953125 0.3953125 0.27265625 0.20104166666666667 0.63359375 0.453125
0 0.288671875 0.35885416666666664 0.16484375 0.17395833333333333 0.36015625 0.41458333333333336
0 0.507421875 0.1359375 0.10703125 0.271875 0.49296875 0.25104166666666666
0 0.59765625 0.07864583333333333 0.090625 0.15729166666666666 0.58203125 0.10833333333333334
```

In [4]:
base_path = "../../../db/object-detection"
df = pd.read_csv(os.path.join(base_path, "annotation.csv"))
df.head()

,label,occluded,truncated,not_ground,occupied,occupied_by_person,ID,xtl,ytl,xbr,ybr,x,y,img_id,id
0,meeting_room_table,True,False,False,False,False,8,51.60,186.10,307.89,332.61,172.16,260.70,8e945fd6-8327-42f6-bfd1-b828a4a82dff,23bef311-d40e-44c3-8dee-bf24118ea02a
1,bag,False,False,False,False,False,9,346.45,165.42,393.28,220.25,369.91,191.48,8e945fd6-8327-42f6-bfd1-b828a4a82dff,ece73ea1-9859-4c7c-a843-30a4c68762e6
2,bag,True,False,False,False,False,10,329.10,133.89,369.63,170.60,350.15,150.37,8e945fd6-8327-42f6-bfd1-b828a4a82dff,e429a150-1202-42a7-9183-c1e22b9fe01c
3,laptop_closed,False,False,False,False,False,19,224.38,287.20,280.56,324.61,253.60,305.88,8e945fd6-8327-42f6-bfd1-b828a4a82dff,34fe393f-b796-45c1-8d9d-b8f2372d6726
4,laptop_opened,False,False,False,False,False,18,197.42,191.27,249.80,228.00,224.38,210.00,8e945fd6-8327-42f6-bfd1-b828a4a82dff,db41ee80-1bd3-4b93-9178-f152342ad08a


In [5]:
df_imgs = pd.read_csv(os.path.join(base_path, "images.csv"))
df_imgs.head()

,id,name,width,height,batch,folder
0,8e945fd6-8327-42f6-bfd1-b828a4a82dff,Cam3_H243_discount_14.png,640.0,480.0,batch1_10.5,batch1
1,3ff5c2b1-83a0-412d-b609-d666d8564393,Cam3_H243_discount_15.png,640.0,480.0,batch1_10.5,batch1
2,ad5018c9-ad3f-49c1-aacd-ad1204f4aff1,Cam3_H243_discount_16.png,640.0,480.0,batch1_10.5,batch1
3,a2e5ba2f-44e2-4b63-83e8-15e637f3a144,Cam3_H243_discount_17.png,640.0,480.0,batch1_10.5,batch1
4,6973f7f5-bf35-4293-a28f-37304a5999e7,Cam3_H243_discount_18.png,640.0,480.0,batch1_10.5,batch1


Algo:
1. Read the annotaion and images;
2. Filter: remove: images without annotations, with errors, duplicates;
3. Select specific classes, merge if needed;
4. Map labels to class ids;
5. Convert to list of dict:
```python
{
    "id",
    "path",
    "objects": {
        "id",
        "cls",
        "bbox",
        "position"
    }
}
```
6. Split on train and val
7. Create YOLO dataset:
    1. label txt
    2. copy images
    3. create project.yaml file

In [6]:
print("before", df_imgs.shape, df.shape)

# rm images without annotations
tmp_df = pd.merge(df, df_imgs, how="outer", left_on="img_id", right_on="id")
tmp_ids = tmp_df[tmp_df["id_x"].isna()]["id_y"].values
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]

# rm images with problems in annotations
tmp_ids = df[(df["xtl"].isna()) | (df["x"].isna())]["img_id"].unique()
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
df = df[~df["img_id"].isin(tmp_ids)]

#rm duplicates
df1 = df_imgs.groupby(['name','batch', "folder", "id"]).size().reset_index()[['name','batch', "folder", "id"]]
duplicates = df1[df1.duplicated('name', keep=False)]
tmp_ids = duplicates.iloc[1::2]["id"].values
df_imgs = df_imgs[~df_imgs["id"].isin(tmp_ids)]
df = df[~df["img_id"].isin(tmp_ids)]

print("after", df_imgs.shape, df.shape)

before (1032, 6) (21595, 15)
after (910, 6) (20531, 15)


In [7]:
def create_project_file(file_pth: str, dataset_pth: str, label_mapping: dict) -> None:
    data = {
        "path": dataset_pth,
        "train": "images/train",
        "val": "images/val",
        "kpt_shape": [1, 2],
    }

    with open(file_pth, "wt") as f:
        yaml.dump(data, f, sort_keys=False, default_flow_style=None)
    
    with open(file_pth, "at") as f:
        yaml.dump({"names": label_mapping}, f, sort_keys=False)

In [8]:
def create_label_txt(args) -> None:
    pth, obj = args
    with open(os.path.join(pth, f"{obj["id"]}.txt"), "wt") as f:
        for det in obj["objects"]:
            w = det["xbr"] - det["xtl"]
            h = det["ybr"] - det["ytl"]
            x_c = det["xtl"] + w / 2
            y_c = det["ytl"] + h / 2
            f.write(f"{det['cls']} {x_c} {y_c} {w} {h} {det['x']} {det['y']}\n")

def cp_img(args) -> None:
    base_pth, pth, obj = args
    shutil.copy(
        os.path.join(base_pth, obj["path"]),
        os.path.join(pth, f"{obj['id']}{os.path.splitext(obj['path'])[-1]}")
    )


def split(objects: list, val_size: float, random_state: int) -> tuple:
    n = len(objects)
    n_val = int(n * val_size)
    idxs = np.arange(n)
    np.random.seed(random_state)
    np.random.shuffle(idxs)
    train_idxs = idxs[:-n_val]
    val_idxs = idxs[-n_val:]
    img_ids = np.array([i['id'] for i in objects])
    return img_ids[train_idxs], img_ids[val_idxs]


def create_yolo_dataset(
        objects: list, label_mapping: dict, base_imgs_pth: str, dataset_pth: str,
        val_size: float = 0.1, random_state: int = 42
) -> None:
    """_summary_

    Args:
        objects (list): list of objects. Object is an image with annotation:
        {
            "id": str,
            "path": str,
            "objects": List[dict]: [
                {
                    "id": str,
                    "cls": int,
                    "xtl": float,
                    "ytl": float,
                    "xbr": float,
                    "ybr": float,
                    "x": float,
                    "y": float
                }
            ]
            
        }.
        label_mapping (dict): labels mapping, e.g. {0: desk, 1: chair}.
        base_imgs_pth (str): _description_
        dataset_pth (str): _description_
        val_size (float, optional): _description_. Defaults to 0.1.
        random_state (int, optional): _description_. Defaults to 42.
    """
    imgs_pth = os.path.join(dataset_pth, "images")
    labels_pth = os.path.join(dataset_pth, "labels")

    shutil.rmtree(dataset_pth, ignore_errors=True)
    os.makedirs(dataset_pth)
    os.makedirs(imgs_pth)
    os.makedirs(os.path.join(imgs_pth, "train"))
    os.makedirs(os.path.join(imgs_pth, "val"))
    os.makedirs(labels_pth)
    os.makedirs(os.path.join(labels_pth, "train"))
    os.makedirs(os.path.join(labels_pth, "val"))

    train_ids, val_ids = split(objects, val_size, random_state)
    train_objects = list(filter(lambda x: x["id"] in train_ids, objects))
    val_objects = list(filter(lambda x: x["id"] in val_ids, objects))

    thread_map(create_label_txt,
               zip([os.path.join(labels_pth, "train")] * len(train_ids), train_objects), desc="train labels")
    thread_map(create_label_txt,
               zip([os.path.join(labels_pth, "val")] * len(val_ids), val_objects), desc="val labels")

    thread_map(
        cp_img,
        zip([base_imgs_pth] * len(train_ids), [os.path.join(imgs_pth, "train")] * len(train_ids), train_objects),
        desc="train images", max_workers=4
    )
    thread_map(
        cp_img,
        zip([base_imgs_pth] * len(val_ids), [os.path.join(imgs_pth, "val")] * len(val_ids), val_objects),
        desc="val images", max_workers=4
    )

    create_project_file(os.path.join(dataset_pth, "project.yaml"), dataset_pth, label_mapping)

In [9]:
def dataframes2objects(df_annotations: pd.DataFrame, df_images: pd.DataFrame) -> tuple:
    labels = df_annotations["label"].unique()
    label_mapping = {idx: i for idx,i in enumerate(labels)}
    reverse_label_mapping = {i: idx for idx,i in enumerate(labels)}
    
    objects = []
    for _, row in df_images.iterrows():
        pth = os.path.join(row["batch"], row["folder"], row["name"])
        dets = []
        for _, row_det in df_annotations[df_annotations["img_id"] == row["id"]].iterrows():
            dets.append({
                "id": row_det["id"],
                "cls": reverse_label_mapping[row_det["label"]],
                "xtl": row_det["xtl"] / row["width"],
                "ytl": row_det["ytl"] / row["height"],
                "xbr": row_det["xbr"] / row["width"],
                "ybr": row_det["ybr"] / row["height"],
                "x": row_det["x"] / row["width"],
                "y": row_det["y"] / row["height"],
            })
        objects.append({
            "id": row["id"],
            "path": pth,
            "objects": dets,
        })
    
    return objects, label_mapping

In [21]:
df_imgs[df_imgs["name"] == "PG1_DL_H250_Office_DESK_lineShirt_26_11_04_12_2017.avi_snapshot_00.00.204.bmp"]

,id,name,width,height,batch,folder
310,4eb3d965-8b6f-48e2-a58d-f89b1e33ffc5,PG1_DL_H250_Office_DESK_lineShirt_26_11_04_12_2017.avi_snapshot_00.00.204.bmp,1280.0,960.0,batch3_21.5,12


In [22]:
objects, label_mapping = dataframes2objects(
    df[df["label"] == "desk_FULL"], df_imgs[df_imgs["name"] == "PG1_DL_H250_Office_DESK_lineShirt_26_11_04_12_2017.avi_snapshot_00.00.204.bmp"])

In [23]:
create_yolo_dataset(objects, label_mapping, base_path, os.path.join(os.getcwd(), "tmp"))

train labels: 0it [00:00, ?it/s]
val labels: 1it [00:00, 18978.75it/s]
train images: 0it [00:00, ?it/s]
val images: 1it [00:00, 174.46it/s]


In [24]:
model = YOLO("yolov8n-pose.yaml")
# Train the model
results = model.train(data="tmp/project.yaml", epochs=25, imgsz=640, device=0, project=os.getcwd(), name="results", exist_ok=True)

Ultralytics YOLOv8.2.25 🚀 Python-3.12.3 torch-2.3.0+cu121 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 7972MiB)
engine/trainer: task=pose, mode=train, model=yolov8n-pose.yaml, data=tmp/project.yaml, epochs=25, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=/home/kyamkovyi/projects/researches/object-detection-v3, name=results, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=

train: Scanning /home/kyamkovyi/projects/researches/object-detection-v3/tmp/labels/val... 1 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1/1 [00:00<00:00, 1185.50it/s]

train: New cache created: /home/kyamkovyi/projects/researches/object-detection-v3/tmp/labels/val.cache
WARNING ⚠️ No 'flip_idx' array defined in data.yaml, setting augmentation 'fliplr=0.0'



val: Scanning /home/kyamkovyi/projects/researches/object-detection-v3/tmp/labels/val.cache... 1 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1/1 [00:00<?, ?it/s]


Plotting labels to /home/kyamkovyi/projects/researches/object-detection-v3/results/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 63 weight(decay=0.0), 73 weight(decay=0.0005), 72 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/kyamkovyi/projects/researches/object-detection-v3/results
Starting training for 25 epochs...

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size


...

      25/25     0.373G       3.07    0.06487          0      2.933      4.489          7        640: 100%|██████████| 1/1 [00:00<00:00, 16.69it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 45.16it/s]

                   all          1          7    0.00699      0.286    0.00826     0.0016     0.0245          1      0.416      0.362



25 epochs completed in 0.025 hours.
Optimizer stripped from /home/kyamkovyi/projects/researches/object-detection-v3/results/weights/last.pt, 6.4MB
Optimizer stripped from /home/kyamkovyi/projects/researches/object-detection-v3/results/weights/best.pt, 6.4MB

Validating /home/kyamkovyi/projects/researches/object-detection-v3/results/weights/best.pt...
Ultralytics YOLOv8.2.25 🚀 Python-3.12.3 torch-2.3.0+cu121 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 7972MiB)
YOLOv8n-pose summary (fused): 187 layers, 3077465 parameters, 0 gradients


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00, 80.88it/s]


                   all          1          7    0.00685      0.286    0.00604    0.00161      0.024          1      0.584      0.482
Speed: 0.4ms preprocess, 6.7ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /home/kyamkovyi/projects/researches/object-detection-v3/results
